# BTS Digital Twin - Round1 P0 Restore Verify

Notebook nay dung de kiem tra `P0` cho `round1 public_set`.

Muc tieu:
- clone repo pipeline
- tu do dataset `round1 public_set`
- audit file size anh train/test
- verify `PASS/FAIL` sau khi da restore du lieu sach

Luu y:
- notebook nay khong hardcode token
- neu repo private, tao Kaggle Secret ten `GITHUB_TOKEN`


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())


## Bước 1 - Clone repo pipeline


In [ ]:
REPO_URL = 'https://github.com/ThongLuc2k3/BTS-Digital-Twin.git'
GIT_BRANCH = 'coordination/round1-status'
GITHUB_TOKEN = ''

try:
    from kaggle_secrets import UserSecretsClient
    if not GITHUB_TOKEN:
        GITHUB_TOKEN = UserSecretsClient().get_secret('GITHUB_TOKEN')
        print('Da lay GITHUB_TOKEN tu Kaggle Secrets')
except Exception:
    pass

clone_url = REPO_URL
if GITHUB_TOKEN and 'github.com' in REPO_URL:
    clone_url = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@')

%cd /kaggle/working
!rm -rf /kaggle/working/project
!git clone --depth 1 -b "{GIT_BRANCH}" "{clone_url}" /kaggle/working/project
%cd /kaggle/working/project


## Bước 2 - Tự dò dataset round1 public_set


In [ ]:
DATASET_ROOT_OVERRIDE = ''

import os
from pathlib import Path

expected = {'hcm0031', 'hcm0034', 'HCM0181', 'HCM0193', 'HCM0204'}
candidates = []
if DATASET_ROOT_OVERRIDE:
    candidates.append(Path(DATASET_ROOT_OVERRIDE))
candidates.append(Path('/kaggle/working/project/Dataset/VAI_NVS_DATA/phase1/public_set'))

for base in [Path('/kaggle/input'), Path('/kaggle/working')]:
    if base.exists():
        for p in base.rglob('public_set'):
            try:
                names = {x.name for x in p.iterdir() if x.is_dir()}
            except Exception:
                continue
            if expected <= names:
                candidates.append(p)

DATASET_ROOT = None
for p in candidates:
    if p.is_dir():
        try:
            names = {x.name for x in p.iterdir() if x.is_dir()}
        except Exception:
            continue
        if expected <= names:
            DATASET_ROOT = str(p)
            break

assert DATASET_ROOT, 'Khong tim thay Dataset/VAI_NVS_DATA/phase1/public_set'
os.environ['DATASET_ROOT'] = DATASET_ROOT
print('DATASET_ROOT =', DATASET_ROOT)


## Bước 3 - Audit round1 public_set hien tai


In [ ]:
import subprocess
from pathlib import Path

PROJECT = Path('/kaggle/working/project')
WORK_DIR = PROJECT / 'pipeline' / 'work'
WORK_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run([
    'python3', str(PROJECT / 'pipeline' / 'scripts' / 'audit_round1_public_images.py'),
    '--dataset_root', DATASET_ROOT,
    '--out_csv', str(WORK_DIR / 'p0_round1_public_audit.csv'),
], check=True)


## Bước 4 - Verify PASS/FAIL sau khi da restore du lieu sach

Neu dang chay tren bo du lieu cu bi hong, cell nay se `FAIL` la dung.
Sau khi ban thay dataset bang bo sach, chay lai cell nay. Chi khi `PASS` moi duoc chuyen sang `M0`.


In [ ]:
import subprocess
from pathlib import Path

PROJECT = Path('/kaggle/working/project')
WORK_DIR = PROJECT / 'pipeline' / 'work'

subprocess.run([
    'python3', str(PROJECT / 'pipeline' / 'scripts' / 'verify_round1_public_restore.py'),
    '--dataset_root', DATASET_ROOT,
    '--out_csv', str(WORK_DIR / 'p0_round1_public_verify.csv'),
], check=True)


## Bước 5 - File ket qua can xem

- Audit CSV: `/kaggle/working/project/pipeline/work/p0_round1_public_audit.csv`
- Verify CSV: `/kaggle/working/project/pipeline/work/p0_round1_public_verify.csv`

Neu verify `PASS`, buoc tiep theo moi la dung `M0`.


In [ ]:
print('/kaggle/working/project/pipeline/work/p0_round1_public_audit.csv')
print('/kaggle/working/project/pipeline/work/p0_round1_public_verify.csv')
